In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score

# --- 1. ADIM: Veriyi Çekme ---
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)

# --- 2. ADIM: Temizlik ---
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df.dropna(inplace=True)
df.drop('customerID', axis=1, inplace=True)

# --- 3. ADIM: ÖZELLİK MÜHENDİSLİĞİ (Zeki Sütunlar) ---
# Bağlılık/Maliyet oranı
df['Tenure_to_MonthlyCharges'] = df['tenure'] / df['MonthlyCharges']

# Toplam ekstra hizmet sayısı
hizmetler = ['PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
             'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
df['Total_Services_Count'] = (df[hizmetler] != 'No').sum(axis=1)

# --- 4. ADIM: Hedef (y) ve Özellik (X) Ayrımı ---
X = df.drop('Churn', axis=1)
y = df['Churn'].map({'Yes': 1, 'No': 0})

X_encoded = pd.get_dummies(X, drop_first=True)

# --- 5. ADIM: Veriyi Bölme (80 Eğitim - 10 Validasyon - 10 Test) ---
X_train, X_temp, y_train, y_temp = train_test_split(X_encoded, y, test_size=0.2, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

# --- 6. ADIM: SMOTE (Dengeyi Sağlıyoruz) ---
# SMOTE'u tekrar aktif ettik! Sadece eğitim verisine uyguluyoruz.
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

# --- 7. ADIM: YENİ MODEL - XGBoost ---
xgb_model = XGBClassifier(
    random_state=42,
    learning_rate=0.03,    # Düşük öğrenme hızı (daha dikkatli)
    max_depth=5,           # Ağaç derinliği
    n_estimators=300,      # Ağaç sayısı
    subsample=0.8,         # Verinin %80'iyle öğren
    colsample_bytree=0.8,  # Sütunların %80'iyle öğren
    eval_metric='logloss'
)

# Modeli SMOTE'lu, dengeli verilerle eğitiyoruz
xgb_model.fit(X_train_smote, y_train_smote)

# Validasyon setiyle tahmin yapıyoruz
y_pred = xgb_model.predict(X_val)

# --- SONUÇLAR ---
print("\n--- XGBOOST (SMOTE + FEATURE ENGINEERING) BAŞARI KARNESİ ---")
print(classification_report(y_val, y_pred))

accuracy = accuracy_score(y_val, y_pred) * 100
print(f"\nGenel Doğruluk Oranı (Accuracy): %{accuracy:.2f}")


--- XGBOOST (SMOTE + FEATURE ENGINEERING) BAŞARI KARNESİ ---
              precision    recall  f1-score   support

           0       0.85      0.84      0.84       516
           1       0.57      0.60      0.59       187

    accuracy                           0.77       703
   macro avg       0.71      0.72      0.72       703
weighted avg       0.78      0.77      0.78       703


Genel Doğruluk Oranı (Accuracy): %77.38
